### Tutorial 3 — Cell segmentation & feature extraction with CORAL

**CORAL is patch-centric**, but it handles cells just as easily by boxing a small patch around each cell. This tutorial takes the cHL CODEX slide you ingested in [Tutorial 1](1-Tissue-Ingest.ipynb) and runs the **cell** pipeline end to end:

1. **Cell segmentation** — get a cell instance mask, two ways: run **Cellpose**, or **import an existing mask** (here, the one that ships with the dataset — which also comes with matched cell-type labels).
2. **Cell-centered patches** — one patch per cell, boxed on its centroid.
3. **Per-cell feature extraction** — a per-cell readout with the mean-marker baseline **and** KRONOS2.

> **Prerequisites.** Run [Tutorial 0](0-Example-Data-Download.ipynb) (data), [Tutorial 1](1-Tissue-Ingest.ipynb) (ingest), and [Tutorial 2](2-Step-by-Step-Patch-Feature-Extraction.ipynb) (tissue) first — this notebook reopens the slide store Tutorial 1 wrote and, for the Cellpose path, needs the tissue mask from Tutorial 2. **Launch Jupyter from the `tutorials/` directory**, and run the `coral` commands below from a terminal at the repo root (`CORAL/`).

Like Tutorials 1 and 2, this notebook is **driven by the `coral` command line**. Each step is one `coral` command (run in a terminal); the notebook only steps in with Python where the CLI genuinely needs a file prepared for it — writing the label CSV and the marker panel — and to display each result. The CLI runs every step over a **whole cohort**: each `.zarr` store in a **job directory** (here `processed/`, the folder Tutorial 1 wrote `raw_image.zarr` into). A final section shows the equivalent single-slide **Python** (`CoralSlide`) API.

| Step                 | CLI                                          | Python (final section)             |
| -------------------- | -------------------------------------------- | ---------------------------------- |
| Segment (Cellpose)   | `coral cell`                                   | `slide.segment_cells(...)`         |
| Import mask + labels | `coral cell --custom-mask-path/--custom-label-path` | `slide.import_cell_mask/labels(...)` |
| Cell patches         | `coral patch --mode cell`                      | `slide.extract_patches(cfg)`       |
| Features             | `coral extract --subset`                       | `slide.encode_features(...)`       |
| Status               | `coral status`                                 | `slide.status()`                   |

#### 0 — Installation

```bash
# Base install — the mean-marker readout and mask import work out of the box
uv sync

# Optional: cell segmentation (Cellpose) — Approach A only; a GPU is recommended
uv sync --extra cells

# Optional: KRONOS2 per-cell embeddings (needs a CUDA GPU) — for step 3
uv sync --extra kronos2
```

Importing the dataset's mask (Approach B) and the mean-marker readout need
neither extra. Cellpose and KRONOS2 are opt-in because they pull in heavy,
version-pinned ML stacks.

#### Reopen the slide from Tutorial 1

Cells build on the **same slide store** Tutorial 1 wrote — no re-ingest. Tutorial 1 named that store after its input folder (`raw_image/` → `raw_image.zarr`), so we point `CoralSlide.open` straight at it. Reopening is the one bit of Python setup; everything else is a terminal command whose result we display.

In [ ]:
from pathlib import Path

from coral import CoralSlide

# Run this notebook from the tutorials/ directory (Jupyter's working dir).
DATA_DIR = Path("example-data")
chl_dir = DATA_DIR / "cHL_CODEX"                        # Tutorial 0's raw data
store_path = DATA_DIR / "processed" / "raw_image.zarr"  # Tutorial 1's slide

CELL_SLUG = "cell_0.37mpp_64px"   # cell patch-set id, written by `coral patch`

assert store_path.exists(), (
    f"No ingested slide at {store_path}. Run Tutorials 0-2 first."
)
slide = CoralSlide.open(store_path)
print(slide)
# Approach A confines cells to Tutorial 2's tissue, so tissue must already have
# run on this store — confirm it shows "completed" below.
print("steps:", {s: i["status"] for s, i in slide.status().items()})

#### 1. Cell segmentation

Get a cell instance mask — a 2-D image where each pixel's value is the id of the cell it belongs to (`0` = background). Two ways:

- **Approach A — Cellpose:** segment the slide's nuclear + membrane channels, confined to Tutorial 2's tissue.
- **Approach B — import an existing mask:** skip Cellpose and load a mask you already have.

**Pick one.** We use **Approach B** for the rest of the tutorial because the cHL dataset ships a mask *and* matched cell-type labels — the ground truth cell phenotyping is benchmarked against. Either way, `coral cell` writes a flat `cells/` folder into each store: `cell_mask` (the instance mask — source of truth), `cell_centroids.csv`, and a `cell_overlay.png` review image.

##### Approach A — Cellpose, confined to tissue

`coral cell` runs a segmenter over the **nuclear** channel (fixed at ingest) plus a **composite membrane** channel (the structural markers, summed), restricted to the tissue mask from Tutorial 2. It needs the `cells` extra and is much faster on a GPU:

```bash
uv run coral cell --job-dir tutorials/example-data/processed --gpu 0
```

Tune with `--cell-diameter`, `--min-cell-size`, or `--membrane-markers CD45,...`; add `--no-tissue` to segment the whole image. Cellpose **relabels** cells with its own ids, so it has no matched labels — for the annotated path, use Approach B.

> **Optional.** No GPU or `cells` extra? **Skip to Approach B.**

##### Approach B — import the dataset's mask and its matched labels

The dataset ships a segmentation mask and a per-cell annotation table. `coral cell --custom-mask-path` takes a **directory** of instance masks named after each store (`raw_image.tiff` for `raw_image.zarr`) — Tutorial 0 named the mask exactly that, so the dataset's own `segmentation/` folder is already a valid mask directory. Imported ids are preserved as-is (no relabel), and the mask is **authoritative**: not tissue-restricted, and it needs neither the `cells` extra nor a GPU.

Labels attach in the **same** command via `--custom-label-path`, a directory of per-store CSVs already in CORAL's `cell_id,label` schema. The raw annotation isn't in that schema yet — its columns are `cellLabel`/`cellType` — so the cell below maps them and writes `cHL_CODEX/labels/raw_image.csv`, giving the CLI a ready label directory to pair with the store.

In [ ]:
import pandas as pd

# The one bit of Python the CLI needs here: stage the labels. `coral cell
# --custom-label-path` wants a directory of per-store CSVs in CORAL's cell_id,label
# schema, named after each store. Map this dataset's columns and write
# labels/raw_image.csv so it pairs with raw_image.zarr. (Inspect your own file's
# columns first — annotation schemas vary between datasets.)
annot = pd.read_csv(chl_dir / "annotation_csv" / "cHL_CODEX_annotation.csv")
print("annotation columns:", list(annot.columns)[:6], "...")

labels = annot.rename(
    columns={"cellLabel": "cell_id", "cellType": "label"}
)[["cell_id", "label"]]

labels_dir = chl_dir / "labels"
labels_dir.mkdir(exist_ok=True)
labels.to_csv(labels_dir / "raw_image.csv", index=False)   # example-data is git-ignored
print("wrote", labels_dir / "raw_image.csv", "|", len(labels), "cells")
print(labels["label"].value_counts())

With the label CSV staged, import the mask **and** attach labels in one call — mask from the dataset's `segmentation/` folder, labels from the `labels/` folder you just wrote:

```bash
uv run coral cell --job-dir tutorials/example-data/processed --custom-mask-path tutorials/example-data/cHL_CODEX/segmentation --custom-label-path tutorials/example-data/cHL_CODEX/labels --label-set maps
```

`--custom-label-path` requires `--custom-mask-path`, since imported ids only line up with an imported mask. `coral cell` **skips stores whose cells are already done**, so if you ran Approach A first, delete the store's `cells/` folder before running this.

In [ ]:
from IPython.display import display
from PIL import Image

# coral cell wrote a review overlay into the store — cell outlines on the nuclear
# channel, tissue boundary in green. Reflects whichever approach you ran.
display(Image.open(slide.path / "cells" / "cell_overlay.png"))

#### 2. Cell-centered patch extraction

`coral patch --mode cell` creates **one patch per cell**, each a `--patch-size` box centred on the cell's centroid (instead of a tissue-filtered grid as in Tutorial 2). 64 px is ~24 µm here, comfortably enclosing a cell. The patch set is stored under a `cell_…` slug (`cell_<mpp>mpp_<size>px`), so it never collides with Tutorial 2's grid patches:

```bash
uv run coral patch --job-dir tutorials/example-data/processed --mode cell --patch-size 64
```

In [ ]:
# coral patch wrote patches/<slug>/patch_overlay.png — one box per cell.
display(Image.open(slide.path / "patches" / CELL_SLUG / "patch_overlay.png"))

#### 3. Per-cell feature extraction

`coral extract` reads the cell patch set and encodes each patch — but on a cell-centered set CORAL first **isolates each patch to its target cell** (pixels outside the cell are zeroed), so the readout is genuinely *per cell*, not per neighbourhood.

##### Restrict to the phenotyping panel

We encode over the **18 phenotypic markers** — the panel that actually separates cell types — rather than all 49 channels. `coral extract --subset` takes a subset YAML (the same schema as `coral ingest --subset` and Tutorial 4); its `channels` selection picks the markers and the subset filename names the output variant `markers_panel` (after `panel.yaml`), so a panel run sits beside any all-marker run. The cell below writes that panel file — the second and last place this notebook needs Python.

In [ ]:
import yaml

# Write the 18-marker panel the CLI selects with --subset. Same file Tutorial 4
# uses. `channels.include` is a glob list matched against CORAL's canonical
# (lower-cased) marker names — what `slide.markers` prints.
PANEL = [
    "dapi", "cd11b", "cd11c", "cd15", "cd163", "cd20", "cd206", "cd30",
    "cd31", "cd4", "cd56", "cd68", "cd7", "cd8", "cytokeratin", "foxp3",
    "mct", "podoplanin",
]

panel_path = DATA_DIR / "panel.yaml"
panel_path.write_text(
    yaml.safe_dump({"name": "panel", "channels": {"include": PANEL}},
                   sort_keys=False)
)
print("wrote", panel_path)
print(panel_path.read_text())

**Mean-marker baseline.** The mean intensity of each panel marker **over the cell's own pixels** — a classic per-cell expression vector, the standard input to cell phenotyping — with no GPU or weights:

```bash
uv run coral extract --job-dir tutorials/example-data/processed --extractor mean_marker --patches cell_0.37mpp_64px --subset tutorials/example-data/panel.yaml
```

Select the cell patch set by its slug `cell_0.37mpp_64px`; omit `--patches` to encode every patch set on the store.

In [ ]:
# Read the features coral extract wrote — a labeled xarray. Cell-centered sets
# carry a cell_id coord; address the panel run by its variant name
# (markers_panel, from panel.yaml). (rows = cells, cols = the 18 panel markers.)
ds = slide.features("mean_marker", CELL_SLUG, suffix="markers_panel")
ds

##### KRONOS2 per-cell embeddings

Swap the encoder — one flag, same command. On the cell patch set KRONOS2 embeds each isolated cell into a 768-dim vector. This needs the `kronos2` extra, a CUDA GPU, **and** granted access to the KRONOS2 Hugging Face repo:

```bash
uv run coral extract --job-dir tutorials/example-data/processed --extractor KRONOS2 --patches cell_0.37mpp_64px --subset tutorials/example-data/panel.yaml --batch-size 1024 --gpu 0
```

Read the embeddings back the same way — `slide.features("KRONOS2", "cell_0.37mpp_64px", suffix="markers_panel")` → an xarray with dims `(patch, feature)` plus a `cell_id` coord. Check progress any time with `uv run coral status --job-dir tutorials/example-data/processed`.

#### 4. The same pipeline in Python

Everything above ran through the `coral` CLI. The `CoralSlide` API does the *same* steps on a single slide, in-process — reach for it to script custom flows or work interactively. Shown here for reference (no need to run it — you already built everything from the terminal):

```python
from pathlib import Path

import pandas as pd

from coral import CoralSlide
from coral.cells import CellposeSegmenter
from coral.config import PatchConfig
from coral.config.subset import Selection
from coral.features import Kronos2Extractor, MeanMarkerExtractor
from coral.io.masks import read_mask_image

slide = CoralSlide.open(Path("example-data/processed/raw_image.zarr"))

# 1a. Segment cells with Cellpose (confined to Tutorial 2's tissue)...
slide.segment_cells(CellposeSegmenter())

# 1b. ...or import the dataset's mask (authoritative, ids preserved) + labels.
mask = read_mask_image(
    Path("example-data/cHL_CODEX/segmentation/raw_image.tiff")
)
slide.import_cell_mask(mask)
annot = pd.read_csv(
    "example-data/cHL_CODEX/annotation_csv/cHL_CODEX_annotation.csv"
)
labels = annot.rename(
    columns={"cellLabel": "cell_id", "cellType": "label"}
)[["cell_id", "label"]]
slide.import_cell_labels(labels, label_set="maps")

# 2. Cell-centered patches — one box per cell.
cell_cfg = PatchConfig(patch_size=64, mode="cell_centered")
slide.extract_patches(cell_cfg)

# 3. Per-cell features over the 18-marker panel (each cell isolated first).
panel = Selection(include=[
    "dapi", "cd11b", "cd11c", "cd15", "cd163", "cd20", "cd206", "cd30",
    "cd31", "cd4", "cd56", "cd68", "cd7", "cd8", "cytokeratin", "foxp3",
    "mct", "podoplanin",
])
slide.encode_features(MeanMarkerExtractor(), cell_cfg, channels=panel)
kronos = Kronos2Extractor.from_pretrained()          # loads weights once
slide.encode_features(kronos, cell_cfg, channels=panel, batch_size=16)
emb = slide.features(kronos, cell_cfg, channels=panel)   # (patch, feature) + cell_id
```

#### 5. Recap

From the slide you ingested in Tutorial 1 you added a cell mask (computed **or** imported), attached matched cell-type labels, cut one patch per cell, and produced per-cell features with both the mean-marker baseline and KRONOS2 — driven from the command line, every step tracked in the slide's state file:

In [ ]:
{step: info['status'] for step, info in slide.status().items()}

**Where next?**

- **Phenotyping / benchmarking.** The per-cell KRONOS2 embeddings + the ground-truth `label` column are exactly the ingredients for a cell-type classification benchmark. [Tutorial 5](5-Cell-Phenotyping.ipynb) picks up right here: spatial folds, a grid-searched linear probe on these embeddings, and a scored comparison against the mean-marker baseline.
- **Cohorts.** Every command above already runs over a directory of slides — point `--job-dir` at a folder of `.zarr` stores for parallel, resumable, per-slide-tracked runs.
- **Custom segmenters.** Subclass `BaseCellSegmenter` (implement `segment(nuclear, membrane, *, mpp)`) to plug in Mesmer, StarDist, or your own model, then pass it to `segment_cells` exactly like `CellposeSegmenter`.